In [1]:
import pandas as pd

# =========================================================
# LOAD
# =========================================================

baseline = pd.read_csv(
    'Baseline_benchmark_result.csv'
)

residual = pd.read_csv(
    'Residual_benchmark_result.csv'
)

# =========================================================
# CLEAN
# =========================================================

def clean_benchmark(df):

    df = df.copy()

    df['flops'] = (
        df['flops']
        .str.replace(' GFLOPS', '', regex=False)
        .astype(float)
    )

    df['gmacs'] = (
        df['gmacs']
        .str.replace(' GMACs', '', regex=False)
        .astype(float)
    )

    df['params'] = (
        df['params']
        .str.replace(' M', '', regex=False)
        .astype(float)
    )

    df['latency_ms'] = (
        df['latency_ms']
        .astype(float)
    )

    return df


baseline = clean_benchmark(baseline)
residual = clean_benchmark(residual)

# =========================================================
# SUMMARY
# =========================================================

summary = pd.DataFrame({

    'Model': [
        'Baseline',
        'Residual'
    ],

    'Params(M)': [
        baseline['params'].mean(),
        residual['params'].mean()
    ],

    'FLOPs(G)': [
        baseline['flops'].mean(),
        residual['flops'].mean()
    ],

    'GMACs(G)': [
        baseline['gmacs'].mean(),
        residual['gmacs'].mean()
    ],

    'Latency(ms)': [
        baseline['latency_ms'].mean(),
        residual['latency_ms'].mean()
    ]
})

print('\n')
print('=' * 80)
print('BENCHMARK SUMMARY')
print('=' * 80)

print(summary.round(4))

# =========================================================
# DIFFERENCE
# =========================================================

print('\n')
print('=' * 80)
print('RESIDUAL VS BASELINE')
print('=' * 80)

for col in [
    'Params(M)',
    'FLOPs(G)',
    'GMACs(G)',
    'Latency(ms)'
]:

    base = summary.loc[
        summary['Model'] == 'Baseline',
        col
    ].values[0]

    res = summary.loc[
        summary['Model'] == 'Residual',
        col
    ].values[0]

    diff = res - base

    pct = diff / base * 100

    print(
        f'{col:15s} | '
        f'Baseline={base:.4f} | '
        f'Residual={res:.4f} | '
        f'Diff={diff:+.4f} ({pct:+.2f}%)'
    )



BENCHMARK SUMMARY
      Model  Params(M)  FLOPs(G)  GMACs(G)  Latency(ms)
0  Baseline     1.9696   18.5265    9.1703      17.6578
1  Residual     1.9696   18.5265    9.1703      17.3561


RESIDUAL VS BASELINE
Params(M)       | Baseline=1.9696 | Residual=1.9696 | Diff=+0.0000 (+0.00%)
FLOPs(G)        | Baseline=18.5265 | Residual=18.5265 | Diff=+0.0000 (+0.00%)
GMACs(G)        | Baseline=9.1703 | Residual=9.1703 | Diff=+0.0000 (+0.00%)
Latency(ms)     | Baseline=17.6578 | Residual=17.3561 | Diff=-0.3016 (-1.71%)


Residual connection을 추가하였음에도 불구하고 파라미터 수(1.97M), FLOPs(18.53G), GMACs(9.17G)는 기존 NAFNet과 동일하게 유지되었다. 또한 평균 추론 지연시간은 17.66ms에서 17.36ms로 유사한 수준을 보였으며, 구조 변경으로 인한 추가적인 계산 비용은 발생하지 않았다.